# Week 09 Blog Activity: SQL in Python
**DSCI 505 – Business Analytics**

This notebook completes the seven Week 09 practice tasks using SQLite and pandas. All queries run against `my_data.db`, which holds the data from the CSVs

## Setup


In [2]:
# os lets Python change folders; sqlite3 connects to the database.
# pandas (pd) reads the CSV files and shows query results as tables (DataFrames).
import os
import sqlite3
import pandas as pd

# Move into the 'files' subfolder only if we're not already in it.
if os.path.basename(os.getcwd()) != 'data':
    os.chdir('data')

# conn identifies the database connection.
# The tables are created from the CSVs in each task below.
conn = sqlite3.connect('my_data.db')
# Create a cursor object to execute SQL commands
cursor = conn.cursor()

## Task 1 – SELECT: Sales Data
Display `sale_date`, `product`, and `price` from the `sales` table.

In [3]:
# Read sales.csv into the 'sales' table
pd.read_csv('sales.csv').to_sql('sales', conn, if_exists='replace', index=False)

# SELECT chooses which columns to display; FROM names the table.
pd.read_sql_query('''
SELECT sale_date, product, price
FROM sales
''', conn)

,sale_date,product,price
0,9/1/2026,Laptop,999.99
1,9/1/2026,Mouse,24.99
2,9/2/2026,Monitor,199.99
3,9/2/2026,Keyboard,49.99
4,9/3/2026,Laptop,999.99
5,9/3/2026,Headphones,79.99
6,9/4/2026,Mouse,24.99
7,9/4/2026,Monitor,199.99
8,9/5/2026,Laptop,999.99
9,9/5/2026,Keyboard,49.99


## Task 2 – WHERE: Human Resources
Find employees in Chicago with more than 10 years of experience using `WHERE` and `AND`.

In [4]:
# Read hr_employees.csv into the 'employees' table
pd.read_csv('hr_employees.csv').to_sql('employees', conn, if_exists='replace', index=False)

# WHERE filters rows; AND requires both conditions to be true.
pd.read_sql_query('''
SELECT *
FROM employees
WHERE city = 'Chicago'
  AND years_experience > 10
''', conn)

,employee_id,name,department,city,salary,years_experience,full_time
0,1027,Lily,Operations,Chicago,56000,17,Yes


## Task 3 – ORDER BY: Real Estate
Display the three largest properties by square footage using `ORDER BY` and `LIMIT`.

In [5]:
# Read realestate_listings.csv into the 'listings' table
pd.read_csv('realestate_listings.csv').to_sql('listings', conn, if_exists='replace', index=False)

# ORDER BY sorts results; DESC puts the largest first.
# LIMIT restricts the output to the first 3 rows.
pd.read_sql_query('''
SELECT *
FROM listings
ORDER BY sqft DESC
LIMIT 3
''', conn)

,listing_id,neighborhood,property_type,bedrooms,sqft,list_price,days_on_market
0,204,Riverside,Condo,5,3736,979000,71
1,206,Maple Grove,Condo,5,3693,1001000,52
2,209,Maple Grove,House,1,3499,661000,116


## Task 4 – GROUP BY: Marketing
Calculate total weekly impressions using `SUM()` and `GROUP BY`.

In [6]:
# Read marketing_campaigns.csv into the 'campaigns' table
pd.read_csv('marketing_campaigns.csv').to_sql('campaigns', conn, if_exists='replace', index=False)

# SUM() adds up values; GROUP BY creates one row per week.
pd.read_sql_query('''
SELECT week, SUM(impressions) AS total_impressions
FROM campaigns
GROUP BY week
ORDER BY week
''', conn)

,week,total_impressions
0,1,181454
1,2,74972
2,3,193242
3,4,180795
4,5,264615
5,6,229892
6,7,235470
7,8,253354


## Task 5 – HAVING: Restaurant
Find dates when total restaurant sales exceeded $400.

In [7]:
# Read restaurant_orders.csv into the 'orders' table
pd.read_csv('restaurant_orders.csv').to_sql('orders', conn, if_exists='replace', index=False)

# GROUP BY totals sales for each date.
# HAVING filters groups after aggregation (WHERE cannot use SUM()).
pd.read_sql_query('''
SELECT order_date,
       ROUND(SUM(order_total), 2) AS total_sales
FROM orders
GROUP BY order_date
HAVING SUM(order_total) > 400
ORDER BY total_sales DESC
''', conn)

,order_date,total_sales
0,2026-10-05,850.10
1,2026-10-12,633.98
2,2026-10-02,492.14
3,2026-10-07,415.94


## Task 6 – Aggregate Functions: Healthcare
Find the average patient age and longest waiting time for each doctor.

In [8]:
#Read clinic_visits.csv into the 'visits' table
pd.read_csv('clinic_visits.csv').to_sql('visits', conn, if_exists='replace', index=False)

# AVG() finds the average and MAX() finds the largest value for each doctor.
pd.read_sql_query('''
SELECT doctor,
       ROUND(AVG(patient_age), 1) AS avg_patient_age,
       MAX(wait_minutes) AS longest_wait_minutes
FROM visits
GROUP BY doctor
ORDER BY doctor
''', conn)

,doctor,avg_patient_age,longest_wait_minutes
0,Dr. Adams,47.0,75
1,Dr. Chen,57.6,50
2,Dr. Gomez,33.7,75
3,Dr. Patel,45.4,59
4,Dr. Scott,44.9,63


## Task 7 – JOIN: Banking
Join the customer and transaction tables to calculate total transaction amounts by branch.

In [9]:
# Read bank_transactions.csv and bank_customers.csv each to their own tables - 'transactions' and 'customers', respectively
pd.read_csv('bank_transactions.csv').to_sql('transactions', conn, if_exists='replace', index=False)
pd.read_csv('bank_customers.csv').to_sql('customers', conn, if_exists='replace', index=False)

# INNER JOIN matches rows from both tables using customer_id.
pd.read_sql_query('''
SELECT customers.branch,
       COUNT(transactions.transaction_id) AS num_transactions,
       ROUND(SUM(transactions.amount), 2) AS total_amount
FROM customers AS customers
INNER JOIN transactions AS transactions
    ON customers.customer_id = transactions.customer_id
GROUP BY customers.branch
ORDER BY total_amount DESC
''', conn)

,branch,num_transactions,total_amount
0,Northgate,12,19546.49
1,Airport,12,15177.98
2,Westside,10,13996.30
3,Main St,3,3581.19


## Close Connection

In [10]:
conn.close()